# HYDRA experiments

Choose an experiment, check the runs it will start, then launch them on **HYDRA at TU Berlin**
using Submitit.
Use the project's Python environment. For a small local example, start with
[introductory.ipynb](introductory.ipynb).


## 1. Load the project

Local `.env` settings are loaded automatically. Set `WANDB_MODE` to `online`,
`offline` or `disabled`. Use `RESULTS_ROOT` to change where results are saved.


In [ ]:
from datetime import datetime, timezone
from uuid import uuid4

import yaml
from dotenv import load_dotenv

from tfm_finetuning.experiments.execution.bundle import write_bundle
from tfm_finetuning.experiments.execution.planning import build_plan
from tfm_finetuning.experiments.execution.submission import submit_plan
from tfm_finetuning.experiments.execution.launch import show_plan
from tfm_finetuning.utils.config import read_yaml_mapping
from tfm_finetuning.utils.paths import get_paths, get_repo_root

root = get_repo_root()
load_dotenv(root / ".env", override=False)
results = (root / get_paths().results_root).resolve()


## 2. Choose an experiment and preview its runs

Set `EXPERIMENT_CONFIG` to the experiment’s YAML file, using a path relative to the
project root. The example selects [distribution_shift_label_noise.yaml](../configs/experiments/distribution_shift_label_noise.yaml).
The file already defines the datasets, models, preprocessing options and training settings.

To add datasets or change model settings, preprocessing options or training seeds,
edit the YAML file. To preserve the original experiment, make a copy, edit it and
point `EXPERIMENT_CONFIG` to the copy. New datasets belong in the group with the
appropriate GPU, memory and time requirements.

`DATASETS`, `MODELS` and `PREPROCESSING` below only select choices already defined
in the file. `None` keeps the file’s selection. They do not add or override settings.
The launch record saves the configuration used for the run.

`SCOPE = "full"` keeps the experiment’s training settings. `"smoke"` reduces the number
of splits and seeds and uses two epochs for ordinary training as a short execution check.
Transfer experiments retain their configured training-update counts.
`BACKEND` chooses where the runs execute, and `SLURM_CONFIG` supplies SLURM settings.

For HYDRA, choose `BACKEND = "slurm"`. The included [hydra_cluster.yaml](../configs/slurm/hydra_cluster.yaml)
contains settings specific to HYDRA at TU Berlin. For another SLURM setup, edit
this file or a copy to match its partitions, GPU constraints, account and resource limits,
then set `SLURM_CONFIG` to that file. Run the kernel on a host with scheduler access and paths
visible to the compute nodes. Jobs use the notebook kernel’s Python environment,
which must also be accessible on those nodes. `"local"` runs on this machine. Previewing launches nothing.


In [ ]:
EXPERIMENT_CONFIG = root / "configs/experiments/distribution_shift_label_noise.yaml"

# Optional filters of the choices in EXPERIMENT_CONFIG. None keeps its selection.
DATASETS = ["jm1"]
MODELS = ["tabpfnv25"]
PREPROCESSING = None
SCOPE = "full"
BACKEND = "local"  # Use "slurm" to submit jobs on HYDRA
SLURM_CONFIG = root / "configs/slurm/hydra_cluster.yaml"  # SLURM settings. Adapt for other SLURM setups than HYDRA

if not EXPERIMENT_CONFIG.is_file():
    raise FileNotFoundError(EXPERIMENT_CONFIG)
experiment_name = EXPERIMENT_CONFIG.stem

if BACKEND not in {"local", "slurm"}:
    raise ValueError("BACKEND must be local or slurm")
cluster = read_yaml_mapping(SLURM_CONFIG) if BACKEND == "slurm" else None
launch_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8]
plan = build_plan(
    root, experiment_name, cluster,
    results / "launches" / f"{experiment_name}-{BACKEND}-{launch_id}",
    datasets=DATASETS, models=MODELS, preprocessing=PREPROCESSING,
    scope=SCOPE, results_root=results, config_path=EXPERIMENT_CONFIG,
)
show_plan(plan)


The next cell shows the first run’s complete settings, combining the selected YAML
file with the model’s defaults. After editing the YAML or changing the filters above,
rerun the preview to update the plan.


In [ ]:
print(f"Experiment configuration: {EXPERIMENT_CONFIG}")
run = plan["jobs"][0]["runs"][0]
print(yaml.safe_dump(run["config"], sort_keys=False))


## 3. Launch

Set `SUBMIT = True` and run this cell. SLURM returns after submission. Local execution
waits for each job. Rerunning the same plan is blocked. Preview again for a new launch.

Datasets are prepared automatically. For TabPFN weights, first run
`uv run python -m tfm_finetuning.data.download.tabpfn_checkpoints` with Hugging Face access configured.
For transfer studies, finish the [required training stages](#Transfer-prerequisites)
first. After partial submission failure, inspect the launch's `jobs.json` before retrying.


In [ ]:
SUBMIT = False  # Set True to execute the previewed plan.

if SUBMIT:
    write_bundle(plan)
    jobs = submit_plan(plan)
    submitted_plan = plan
    print(f"Launched {len(jobs)} jobs. Records: {plan['output']}/jobs.json")
else:
    print("Preview only. Set SUBMIT = True in this cell to launch the selected plan.")

## 4. Check jobs

Rerun the status cell to refresh. The next cell shows the first job's logs.
`jobs[i].result()` waits for a result. `jobs[i].cancel()` cancels that job.


In [ ]:
if "jobs" in globals():
    print(f"Launch: {submitted_plan['output']}")
    display([{"job_id": job.job_id, "state": job.state, "done": job.done()} for job in jobs])
else:
    print("No jobs submitted in this kernel yet.")

In [ ]:
if "jobs" in globals() and jobs:
    job = jobs[0]  # Select another index to inspect a different run.
    print(f"stdout: {job.paths.stdout}\nstderr: {job.paths.stderr}")
    if job.paths.stdout.exists():
        print(job.stdout())
    if job.paths.stderr.exists():
        print(job.stderr())

Results are saved under `results/<study>/<run_id>/` (or the configured `RESULTS_ROOT`).
Launch folders record plans, job IDs and logs. Completed runs are skipped on rerun.
Browse the [experiment configurations](../configs/experiments) or [available scripts](../scripts/experiments)
for the available experiments.


## Optional terminal commands

The scripts in [scripts/experiments](../scripts/experiments) are Bash shortcuts for the
same Python launcher. They are grouped into `overview`, `transformations`, `transfer`
and `appendix`. For example, from the project root:

```bash
uv run python -m tfm_finetuning.experiments.execution.launch distribution_shift_label_noise \
  --local --dataset jm1 --model mitra --dry-run
```

The equivalent shortcut is `uv run bash scripts/experiments/transformations/label_noise.sh`
with the same options. Replace `--dry-run` with `--run` for local execution, or use
`--cluster configs/slurm/hydra_cluster.yaml --submit` for SLURM. Run the SLURM command
from the same configured environment described above. Shortcuts preview by default.

[scripts/reproduce_local.sh](../scripts/reproduce_local.sh) runs the complete local
sequence, including the optional analyses. Developer checks are under
[scripts/development](../scripts/development).


## Transfer prerequisites

Transfer runs keep their configured update budgets, preprocess datasets independently
and select checkpoints using validation data. Pooled training alternates dataset-specific
episodes. Each sequential stage starts a fresh optimizer from the preceding stage's
selected weights. Context evaluations change the labeled ICL context without further
training.

Finish the required training stages before starting experiments that reuse their weights:

| Run first | Then run |
| --- | --- |
| [Training order](../configs/experiments/transfer.yaml) | [Target-data budgets](../configs/experiments/dataset_transfer_data_budget.yaml), [balanced classification budgets](../configs/experiments/dataset_transfer_balanced_budget_classification.yaml), [source-only classification](../configs/experiments/dataset_transfer_source_only_eval_classification.yaml) |
| [Houses source checkpoint](../configs/experiments/dataset_transfer_houses_source_checkpoint.yaml) | [Balanced regression budgets](../configs/experiments/dataset_transfer_balanced_budget_regression.yaml), [source-only regression](../configs/experiments/dataset_transfer_source_only_eval_regression.yaml) |
| [Natural-domain training](../configs/experiments/dataset_transfer_natural_domain_training.yaml) | [Natural-domain context evaluation](../configs/experiments/dataset_transfer_natural_domain.yaml) |
| [Retention training](../configs/experiments/dataset_transfer_learned_retention_train.yaml) | [Retention evaluation](../configs/experiments/dataset_transfer_learned_retention_eval.yaml) |

Keep these runs under the same results root. Checkpoints must come from completed runs
with matching model, preprocessing, seed, split, row budgets and training settings.
Missing checkpoint files cause an error; the evaluation does not silently retrain them.
Set `dataset_transfer.checkpoint_path` to select weights manually.

The Bank/Covertype retention comparisons also use the Covertype target-only 100-update
reference from `dataset_transfer` and Bank's 100-update reference from
[dataset_transfer_target_controls](../configs/experiments/dataset_transfer_target_controls.yaml).
Shorter-budget references are selected using validation within their own update cap.
They share an optimization path only when learning-rate scheduling and early stopping
are disabled.
